# Dropout analysis

Participants who dropped out against those who completed the study: pre-survey measures, time of dropout, excluded conversations, guardrail refusals (Section 5, Dropout Reasons).


## 0. Setup


In [1]:
from pathlib import Path

import zlib

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 210)
pd.set_option("display.max_colwidth", 92)

BASE_DIR = (Path(".").resolve() / "../data").resolve()
DB_DIR, FINAL_DIR = BASE_DIR / "database", BASE_DIR / "final"

LEFT, STAYED = "DROPOUT", "COMPLETE"                    # the two `status` values compared
GROUPS = [LEFT, STAYED]
N_BOOT = 10_000
SEED = 20260903

demographics = pd.read_csv(FINAL_DIR / "participant_demographics.csv")
annotations = pd.read_csv(FINAL_DIR / "annotations_export.csv")
interaction = pd.read_csv(FINAL_DIR / "user_interaction.csv", parse_dates=["createdOn"])
retrieval = pd.read_csv(DB_DIR / "chat_retrieval.csv")

STATUS = demographics.set_index("username")["status"]
interaction["status"] = interaction["username"].map(STATUS)

wrote = demographics[demographics["status"] != "NOCHAT"].copy()
assert set(wrote["username"]) == set(annotations["username"]), \
    "the annotation covers a different population than status says wrote a message"

print(pd.crosstab(demographics["condition"], demographics["status"], margins=True))
print(f"\ncompared below: {(wrote['status'] == STAYED).sum()} completed vs. "
      f"{(wrote['status'] == LEFT).sum()} dropouts; "
      f"{(demographics['status'] == 'NOCHAT').sum()} never wrote and are left out")

status     COMPLETE  DROPOUT  NOCHAT  All
condition                                
Assistant        23       11       3   37
Interview        26        7       5   38
All              49       18       8   75

compared below: 49 completed vs. 18 dropouts; 8 never wrote and are left out


### 0.1 Effect-size functions


In [2]:
def hedges_g(a, b):
    """g = (mean(a) - mean(b)) / pooled SD, Hedges-corrected. NaN if the pooled SD is 0."""
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if not pooled > 0:
        return np.nan
    return (a.mean() - b.mean()) / pooled * (1 - 3 / (4 * (na + nb) - 9))


def compare(series, status, label):
    """One row: the two group means and g with a bootstrap CI, dropout minus completed."""
    rng = np.random.default_rng(SEED + zlib.crc32(label.encode()))
    keep = pd.notna(series) & status.isin(GROUPS).values
    x, s = np.asarray(series, dtype=float)[keep], np.asarray(status)[keep]
    a, b = x[s == LEFT], x[s == STAYED]

    draws = np.array([hedges_g(rng.choice(a, len(a), replace=True),
                               rng.choice(b, len(b), replace=True)) for _ in range(N_BOOT)])
    finite = draws[np.isfinite(draws)]
    lo, hi = (np.percentile(finite, [2.5, 97.5]) if len(finite) else (np.nan, np.nan))
    point = hedges_g(a, b)
    return {"measure": label, "n_dropout": len(a), "n_completed": len(b),
            "mean_dropout": a.mean(), "mean_completed": b.mean(),
            "g": point, "ci_low": lo, "ci_high": hi,
            "excludes_0": bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0)),
            "sign_agree": float(np.mean(np.sign(finite) == np.sign(point))) if len(finite) else np.nan}


def effect_table(frame, status, labels=None):
    labels = labels or {}
    return pd.DataFrame([compare(frame[c], status, labels.get(c, c))
                         for c in frame.columns]).set_index("measure")

## 1. Pre-survey measures


In [3]:
AI_ITEMS = [f"QID19_{i}" for i in range(1, 7)]           # AI literacy, -3..3
VAA_ITEMS = ["QID31_1", "QID31_2"]                      # VAAs educate / are a good thing, -2..2
AGE_MIDPOINT = {"18-24": 21, "25-34": 29.5, "35-44": 39.5,
                "45-54": 49.5, "55-64": 59.5, "über 65": 70}

profile = wrote.assign(age=wrote["QID6"].map(AGE_MIDPOINT),
                       ai_literacy=wrote[AI_ITEMS].mean(axis=1),
                       vaa_attitude=wrote[VAA_ITEMS].mean(axis=1),
                       university=wrote["QID13"].eq("Universität oder ETH"))

MEASURES = {"age": "age (band midpoint)", "ai_literacy": "AI literacy (-3..3)",
            "vaa_attitude": "attitude towards VAAs (-2..2)", "QID21": "political interest (0..10)"}
effects = effect_table(profile[list(MEASURES)], profile["status"], MEASURES)
display(effects.round(2))

print("university degree, the one categorical measure the subsection reports:")
display(pd.crosstab(profile["status"], profile["university"], margins=True)
        .rename(columns={False: "other", True: "Universität oder ETH"}))

for top in (67.5, 70, 72.5):
    ages = profile["QID6"].map({**AGE_MIDPOINT, "über 65": top})
    g = hedges_g(ages[profile["status"] == LEFT].to_numpy(),
                 ages[profile["status"] == STAYED].to_numpy())
    print(f"open top band read as {top}: mean age {ages[profile['status'] == LEFT].mean():.1f} "
          f"vs {ages[profile['status'] == STAYED].mean():.1f}, g = {g:+.2f}")

,n_dropout,n_completed,mean_dropout,mean_completed,g,ci_low,ci_high,excludes_0,sign_agree
measure,,,,,,,,,
age (band midpoint),18,49,54.67,47.03,0.54,0.01,1.14,True,0.98
AI literacy (-3..3),18,49,0.25,1.04,-0.61,-1.33,0.04,False,0.97
attitude towards VAAs (-2..2),18,49,0.17,0.74,-0.59,-1.21,-0.03,True,0.98
political interest (0..10),18,49,6.56,6.98,-0.17,-0.79,0.40,False,0.72


university degree, the one categorical measure the subsection reports:


university,other,Universität oder ETH,All
status,,,
COMPLETE,39,10,49
DROPOUT,17,1,18
All,56,11,67


open top band read as 67.5: mean age 53.8 vs 46.7, g = +0.53
open top band read as 70: mean age 54.7 vs 47.0, g = +0.54
open top band read as 72.5: mean age 55.5 vs 47.3, g = +0.56


## 2. Time of dropout


In [4]:
votes = pd.read_csv(DB_DIR / "user_vote.csv", dtype=str)
user = pd.read_csv(DB_DIR / "user.csv", dtype=str)
survey = pd.read_csv(DB_DIR / "survey.csv", dtype=str)
CHATBOT_SURVEYS = set(survey.loc[survey["chatbot"] != "none", "id"])

votes["username"] = votes["createdById"].map(dict(zip(user["id"], user["username"])))
chat_votes = votes[votes["surveyId"].isin(CHATBOT_SURVEYS) & votes["username"].notna()]

depth = (interaction.groupby(["username", "status"])
         .agg(statements=("topic", "nunique"),
              written=("role", lambda r: int((r == "user").sum())),
              minutes=("createdOn", lambda s: (s.max() - s.min()).total_seconds() / 60))
         .reset_index())
depth["votes"] = depth["username"].map(chat_votes.groupby("username").size()).fillna(0).astype(int)

print("median per participant")
display(depth.groupby("status")[["written", "minutes"]].median().round(1))

print("\nthe statement the dropouts were on when they left, against the votes they had cast")
left = depth[depth["status"] == LEFT]
display(pd.crosstab(left["statements"], left["votes"], margins=True)
        .rename_axis(index="statements reached", columns="votes cast"))
# Nobody skipped a statement: a dropout had voted on every statement before the one they were
# on, and sometimes on that one too (they voted, then left before writing on the next).
assert (left["statements"] - left["votes"]).isin([0, 1]).all(), \
    "a dropout cast fewer votes than the statements they had already passed"

median per participant


,written,minutes
status,,
COMPLETE,18.0,12.2
DROPOUT,7.0,3.2



the statement the dropouts were on when they left, against the votes they had cast


votes cast,0,1,2,3,All
statements reached,,,,,
1,7,2,0,0,9
2,0,5,0,0,5
3,0,0,2,0,2
4,0,0,0,2,2
All,7,7,2,2,18


## 3. Excluded conversations


In [5]:
conversations = (annotations.groupby(["username", "questionIndex"])
                 .agg(condition=("condition", "first"), excluded=("conversation_excluded", "max"))
                 .reset_index())
conversations["status"] = conversations["username"].map(STATUS)

print(f"{int(conversations['excluded'].sum())} of {len(conversations)} conversations excluded, "
      f"{int(annotations['conversation_excluded'].sum())} of {len(annotations)} user turns")
display(pd.crosstab(conversations["status"], conversations["excluded"], margins=True)
        .rename_axis(columns="excluded"))
display(pd.crosstab(conversations["condition"], conversations["excluded"], margins=True)
        .rename_axis(columns="excluded"))

38 of 229 conversations excluded, 226 of 1098 user turns


excluded,0,1,All
status,,,
COMPLETE,172,24,196
DROPOUT,19,14,33
All,191,38,229


excluded,0,1,All
condition,,,
Assistant,97,14,111
Interview,94,24,118
All,191,38,229


## 4. Guardrail refusals by group


In [6]:
violated = retrieval["guardrailViolated"].astype(str).str.lower() == "true"
BLOCK_CATEGORY = retrieval.loc[violated].set_index("userMessageId")["guardrailCategory"].to_dict()

turns = interaction[interaction["role"] == "user"].copy()
turns["category"] = turns["message_id"].map(BLOCK_CATEGORY)
turns["blocked"] = turns["category"].notna()

print(f"{int(turns['blocked'].sum())} of {len(turns)} user turns blocked ({turns['blocked'].mean():.1%}), "
      f"{int(turns.loc[turns['status'] == LEFT, 'blocked'].sum())} of them from dropouts, "
      f"who wrote {turns['status'].eq(LEFT).mean():.0%} of all turns")
display(turns.loc[turns["blocked"], "category"].value_counts().rename("turns").to_frame())

display(turns.groupby("status")["blocked"].agg(blocked="sum", turns="size")
        .assign(share=lambda d: (d["blocked"] / d["turns"]).round(3)))

per = turns.groupby(["username", "status"])["blocked"].agg(blocked="sum", turns="size").reset_index()
per["share"] = per["blocked"] / per["turns"]
display(per.groupby("status").agg(participants=("username", "size"),
                                  any_block=("blocked", lambda s: int((s > 0).sum())),
                                  third_or_more=("share", lambda s: int((s >= 1 / 3).sum()))))

151 of 1098 user turns blocked (13.8%), 38 of them from dropouts, who wrote 14% of all turns


,turns
category,
off_topic,88
llm_opinion,25
critical,22
vote_intention,16


,blocked,turns,share
status,,,
COMPLETE,113,947,0.119
DROPOUT,38,151,0.252


,participants,any_block,third_or_more
status,,,
COMPLETE,49,35,1
DROPOUT,18,9,4


### 4.1 Last chatbot turn before dropout


In [7]:
last_turn = turns.sort_values("createdOn").groupby("username").tail(1)
ended_on_a_block = last_turn[(last_turn["status"] == LEFT) & last_turn["blocked"]]

ended_while_staying = last_turn[(last_turn["status"] == STAYED) & last_turn["blocked"]]
print(f"{len(ended_on_a_block)} of {(last_turn['status'] == LEFT).sum()} dropouts ended on a "
      f"blocked message, against {len(ended_while_staying)} of the "
      f"{(last_turn['status'] == STAYED).sum()} who completed")
display(ended_on_a_block[["condition", "topic", "category", "content", "content_en"]]
        .set_index("category"))

for username in ended_on_a_block["username"]:
    print("\n" + "-" * 100)
    tail = interaction[interaction["username"] == username].sort_values("createdOn").tail(4)
    for _, row in tail.iterrows():
        mark = f"[{BLOCK_CATEGORY.get(row['message_id'], '')}]" if row["role"] == "user" else ""
        print(f"  {row['role']:<9} {mark:<18} {str(row['content'])[:150]}")

4 of 18 dropouts ended on a blocked message, against 0 of the 49 who completed


,condition,topic,content,content_en
category,,,,
off_topic,Interview,Education,Ich sehe keine Schaltfläche,I don't see a button
off_topic,Interview,Energy & transport,Keine,NaN
off_topic,Assistant,Welfare state & family,Autofahren bis 70?,Driving until 70?
llm_opinion,Interview,Welfare state & family,?,?



----------------------------------------------------------------------------------------------------
  user      []                 Gleichfalls
  assistant                    Wir haben unser Gespräch jetzt beendet. Da Sie keine weitere Frage gestellt haben, können Sie über die angezeigte Schaltfläche Ihre Meinung abgeben un
  user      [off_topic]        Ich sehe keine Schaltfläche
  assistant                    Diese Frage liegt ausserhalb meines Themenbereichs. Ich helfe Ihnen gerne bei Fragen zur Schweizer Politik, zu Parteien oder zu Kandidierenden.

----------------------------------------------------------------------------------------------------
  user      [off_topic]        Ich möchte das thema Politik beenden
  assistant                    Diese Frage liegt ausserhalb meines Themenbereichs. Ich helfe Ihnen gerne bei Fragen zur Schweizer Politik, zu Parteien oder zu Kandidierenden.
  user      [off_topic]        Keine
  assistant                    Diese Frage liegt ausserha

## 5. Reported numbers


In [8]:
claims = {
    "75 participants opened the platform": len(demographics) == 75,
    "49 completed (23 assistant, 26 interviewer)":
        (demographics["status"] == STAYED).sum() == 49
        and (demographics.loc[demographics["status"] == STAYED, "condition"]
             .value_counts().to_dict() == {"Interview": 26, "Assistant": 23}),
    "26 dropouts, 8 of whom never wrote":
        (demographics["status"] != STAYED).sum() == 26
        and (demographics["status"] == "NOCHAT").sum() == 8,
    "18 stopped mid-conversation": (demographics["status"] == LEFT).sum() == 18,
    "55 vs 47 years on average":
        (round(profile.loc[profile["status"] == LEFT, "age"].mean()) == 55
         and round(profile.loc[profile["status"] == STAYED, "age"].mean()) == 47),
    "1 of 18 vs 10 of 49 hold a university degree":
        profile.groupby("status")["university"].sum().to_dict() == {STAYED: 10, LEFT: 1},
    "every interval includes zero or ends within 0.05 of it":
        effects.apply(lambda r: (r["ci_low"] <= 0 <= r["ci_high"])
                      or min(abs(r["ci_low"]), abs(r["ci_high"])) <= 0.05, axis=1).all(),
    "9 of 18 stopped during the first statement": int((left["statements"] == 1).sum()) == 9,
    "7 never submitted a rating": int((left["votes"] == 0).sum()) == 7,
    "14 of 33 dropout conversations excluded, against 24 of 196":
        pd.crosstab(conversations["status"], conversations["excluded"]).loc[
            [LEFT, STAYED], 1].tolist() == [14, 24],
    "151 of 1,098 turns blocked, 38 of them from dropouts":
        int(turns["blocked"].sum()) == 151 and len(turns) == 1098
        and int(turns.loc[turns["status"] == LEFT, "blocked"].sum()) == 38,
    "4 of 18 dropouts vs 1 of 49 completers blocked on a third of their turns":
        per.groupby("status")["share"].apply(lambda s: int((s >= 1 / 3).sum())).to_dict()
        == {LEFT: 4, STAYED: 1},
    "4 dropouts ended on a blocked message": len(ended_on_a_block) == 4,
}
for claim, holds in claims.items():
    print(f"  {'ok ' if holds else 'STALE'}  {claim}")
assert all(claims.values()), "the Dropout subsection no longer matches the data"

  ok   75 participants opened the platform
  ok   49 completed (23 assistant, 26 interviewer)
  ok   26 dropouts, 8 of whom never wrote
  ok   18 stopped mid-conversation
  ok   55 vs 47 years on average
  ok   1 of 18 vs 10 of 49 hold a university degree
  ok   every interval includes zero or ends within 0.05 of it
  ok   9 of 18 stopped during the first statement
  ok   7 never submitted a rating
  ok   14 of 33 dropout conversations excluded, against 24 of 196
  ok   151 of 1,098 turns blocked, 38 of them from dropouts
  ok   4 of 18 dropouts vs 1 of 49 completers blocked on a third of their turns
  ok   4 dropouts ended on a blocked message
